[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00


In [3]:
import torch
import math

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

def rope_freq(q, theta):
  b, s, d = q.shape
  inv_freq = 1.0 / (theta**(torch.arange(0, d, 2)/d))
  freq = torch.outer(torch.arange(s), inv_freq)
  return torch.polar(torch.ones_like(freq), freq)

def apply_rope(q, k):
  b, s, d = q.shape
  freq_q = rope_freq(q, 10000)
  freq_k = rope_freq(k, 10000)
  q = torch.view_as_complex(q.reshape(b, s, -1, 2))
  k = torch.view_as_complex(k.reshape(b, s, -1, 2))
  q = q * freq_q.unsqueeze(0)
  k = k * freq_k.unsqueeze(0)
  q = torch.view_as_real(q).flatten(-2)
  k = torch.view_as_real(k).flatten(-2)
  return q, k

In [8]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print(qr.shape)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

torch.Size([1, 8, 16])
Shape preserved: True
Norm preserved: True


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (4.1ms)
  ✅ [2/4] Preserves norm (3.5ms)
  ✅ [3/4] Relative position property (11.3ms)
  ✅ [4/4] Gradient flow (17.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (36.1ms total)
  Progress saved. Run status() to see your dashboard.

